In [1]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your new DBpedia test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ1.json")
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "final_model_mrr_0.4487.pt")

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")

/workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Success! Project root set to: /workspace/SPARQL-Embeddings
Dataset Loaded -> Test Split: 337
Evaluation Ready -> Anchors: 337 | Corpus: 1677


In [2]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# BM25
evaluate_bm25(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen2.5-Coder (generative code LLM used as an embedder, last-token pooling;
# 0.5B ≈ scale-matched to the dense 0.6B Qwen3-Embedding it replaces)
evaluate_code_llm('Qwen/Qwen2.5-Coder-0.5B', 'Qwen2.5-Coder-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)


Qwen3-Emb-0.6B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0267  0.0159  0.0181  0.0425  0.0653    0.0267  0.0264
@5     0.0273  0.0243  0.0226  0.0517  0.1068    0.0273  0.0309
@10    0.0270  0.0371  0.0266  0.0626  0.1899    0.0270  0.0373
@20    0.0276  0.0749  0.0348  0.0708  0.3116    0.0276  0.0522


In [ ]:
# Prior-literature SPARQL / code encoder baselines (zero-shot, mean-pooled):
#   CodeBERT (SparqlCE's own backbone), GraphCodeBERT (CodeBERT + data-flow
#   graph), SPBERT (BERT pretrained on SPARQL). Defined in evaluate_rq1.py as
#   SPARQL_ENCODER_BASELINES. The "mean pooling" notice each emits is correct
#   here (raw MLM encoders), not the LoRA-adapter bug.
evaluate_sparql_encoder_baselines(eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

In [3]:

display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)



SBERT/MiniLM
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0208  0.0101  0.0123  0.0396  0.0593    0.0208  0.0218
@5     0.0196  0.0134  0.0137  0.0445  0.0801    0.0196  0.0231
@10    0.0172  0.0201  0.0154  0.0495  0.1187    0.0172  0.0247
@20    0.0194  0.0447  0.0230  0.0545  0.1929    0.0194  0.0352


In [4]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)



SBERT/MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0307  0.0141  0.0169  0.0589  0.0772    0.0307  0.0339
@5     0.0285  0.0203  0.0202  0.0637  0.0979    0.0285  0.0349
@10    0.0270  0.0328  0.0248  0.0699  0.1424    0.0270  0.0386
@20    0.0269  0.0580  0.0316  0.0762  0.2315    0.0269  0.0487
